## Prerequisites

## Load 'users_001.csv'into dataFrame

In [0]:
df = spark.read.csv(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
display(df)

## [Trasaction 01] - write DataFrame as delta

In [0]:
df.write.format('DELTA').save(path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta")

## [Transaction - 02] Over Write Delta Folder

In [0]:
from pyspark.sql.functions import col
df.filter(col("country") == "India").write.save(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta", mode="OVERWRITE"
)

## Read Delta

In [0]:
spark.read.load(path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta").show()

## [Advantage] - Maintains Versions

In [0]:
spark.read.option("VersionAsOf",0).load(path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta").show()

## Read Transaction log

### Approach 01

In [0]:
spark.read.load(format="text",path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta/_delta_log/00000000000000000000.json").show(truncate=False)

### Approach 02

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark, "/Volumes/wns24082026/quickstart_schema/sandbox/output_delta/"
)
delta_table.history().display()

- Read Delta as DF
- Create a view on DF
- Updates are allowed

In [0]:
df = spark.read.load(path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta")

In [0]:
display(df)

In [0]:
df.createOrReplaceTempView("test_vw")

In [0]:
%sql

UPDATE test_vw
SET country="Bharat"
WHERE country="India"

In [0]:
spark.read.load(path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta").show()